# Demo 01 — DDPM on MNIST from Scratch
### TLH-1 · Applied Models: Diffusion Models

Implements **Denoising Diffusion Probabilistic Models (Ho et al. 2020)** from scratch in PyTorch.
No HuggingFace diffusers — pure math → code. Runs on CPU in ~5 minutes.

**What we build:**
1. Noise schedule (linear β schedule)
2. Forward process `q(x_t | x_0)` — closed-form, sample any timestep directly
3. U-Net: Conv + ResNet blocks + sinusoidal time embedding
4. Training loop: denoising MSE objective
5. DDPM sampling: 1000-step reverse process
6. Visualizations: noise trajectories + generated digits

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Part 1: Noise Schedule

The forward process adds noise according to a schedule $\beta_1 < \beta_2 < \cdots < \beta_T$.

We precompute everything needed:
- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ — cumulative product
- $\sqrt{\bar{\alpha}_t}$ and $\sqrt{1 - \bar{\alpha}_t}$ — for the closed-form forward sample

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
T          = 1000
beta_start = 1e-4
beta_end   = 0.02
img_size   = 28
batch_size = 128
lr         = 2e-4
epochs     = 10
model_dim  = 64

# ── Noise schedule ───────────────────────────────────────────────────────────
betas      = torch.linspace(beta_start, beta_end, T, device=device)   # linear schedule
alphas     = 1.0 - betas
alpha_bar  = torch.cumprod(alphas, dim=0)                              # ᾱ_t
sqrt_ab    = alpha_bar.sqrt()                                          # √ᾱ_t
sqrt_1mab  = (1.0 - alpha_bar).sqrt()                                  # √(1-ᾱ_t)

# ── Visualise ᾱ_t ────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.plot(alpha_bar.cpu(), color='#8f3e22', linewidth=2)
plt.xlabel('timestep t')
plt.ylabel('ᾱ_t')
plt.title('Cumulative noise schedule — signal fraction remaining')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'ᾱ at t=0: {alpha_bar[0]:.4f}   ᾱ at t=999: {alpha_bar[-1]:.4f}')

## Part 2: Forward Process — q(x_t | x_0)

The key insight: instead of iterating through $t$ steps, we sample $x_t$ directly from $x_0$:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

In code: `x_t = sqrt_ab[t] * x0 + sqrt_1mab[t] * noise`

In [ ]:
def extract(a, t, x_shape):
    """Index into 1D schedule tensor at timestep t, broadcast to x_shape."""
    b = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

def q_sample(x0, t, noise=None):
    """Forward process: sample x_t from x_0 in one step."""
    if noise is None:
        noise = torch.randn_like(x0)
    return extract(sqrt_ab, t, x0.shape) * x0 + extract(sqrt_1mab, t, x0.shape) * noise


# ── Load one MNIST batch and visualise the corruption ────────────────────────
dataset = datasets.MNIST(
    root='./data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2 - 1)   # scale to [-1, 1]
    ])
)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
x0_demo, _ = next(iter(loader))
x0_demo    = x0_demo[:8].to(device)

timesteps_to_show = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(len(timesteps_to_show), 8, figsize=(12, len(timesteps_to_show) * 1.5))

for row, t_val in enumerate(timesteps_to_show):
    t_tensor = torch.full((8,), t_val, device=device, dtype=torch.long)
    x_t      = q_sample(x0_demo, t_tensor)
    for col in range(8):
        img = x_t[col, 0].cpu().clamp(-1, 1).numpy()
        axes[row, col].imshow((img + 1) / 2, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f't={t_val}', fontsize=9)

plt.suptitle('Forward process: x_0 → x_T (top to bottom)', y=1.01)
plt.tight_layout()
plt.show()

## Part 3: U-Net Architecture

The denoiser $\epsilon_\theta(x_t, t)$ is a U-Net conditioned on timestep $t$:

```
x_t → [Down×2] → [Bottleneck] → [Up×2 + skips] → ε̂
                      ↑ time embedding injected at every ResBlock
```

**Time embedding:** sinusoidal (same as Transformers) → MLP → added to each ResBlock via shift.

In [ ]:
# ── Time embedding ────────────────────────────────────────────────────────────
def sinusoidal_embedding(t, dim):
    """Sinusoidal positional encoding for timestep t. t: (B,) → (B, dim)"""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
    args  = t[:, None].float() * freqs[None]      # (B, half)
    return torch.cat([args.sin(), args.cos()], dim=-1)   # (B, dim)


# ── Building blocks ───────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1    = nn.GroupNorm(8, in_ch)
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj   = nn.Linear(t_dim, out_ch)   # time → channel shift
        self.norm2    = nn.GroupNorm(8, out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip     = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act      = nn.SiLU()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = h + self.t_proj(self.act(t_emb))[:, :, None, None]  # inject time
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.res  = ResBlock(in_ch, out_ch, t_dim)
        self.down = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, t_emb):
        x = self.res(x, t_emb)
        return self.down(x), x     # return (downsampled, skip)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, t_dim):
        super().__init__()
        self.up  = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.res = ResBlock(in_ch + skip_ch, out_ch, t_dim)

    def forward(self, x, skip, t_emb):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x, t_emb)


# ── U-Net ─────────────────────────────────────────────────────────────────────
class UNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, t_dim=256):
        super().__init__()
        self.t_mlp = nn.Sequential(
            nn.Linear(base_ch, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.init_conv  = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.down1      = DownBlock(base_ch,     base_ch * 2, t_dim)
        self.down2      = DownBlock(base_ch * 2, base_ch * 4, t_dim)
        self.bottleneck = ResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.up1        = UpBlock(base_ch * 4, base_ch * 2, base_ch * 2, t_dim)
        self.up2        = UpBlock(base_ch * 2, base_ch,     base_ch,     t_dim)
        self.out_conv   = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t):
        t_emb = self.t_mlp(sinusoidal_embedding(t, model_dim))   # (B, t_dim)
        x  = self.init_conv(x)
        x, s1 = self.down1(x, t_emb)
        x, s2 = self.down2(x, t_emb)
        x  = self.bottleneck(x, t_emb)
        x  = self.up1(x, s2, t_emb)
        x  = self.up2(x, s1, t_emb)
        return self.out_conv(x)


model     = UNet(in_ch=1, base_ch=model_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
n_params  = sum(p.numel() for p in model.parameters())
print(f'U-Net parameters: {n_params:,}   (~{n_params/1e6:.1f}M)')

## Part 4: Training Loop

The training objective:
$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(\sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon,\; t)\|^2\right]$$

In 10 lines: sample a timestep, add noise, predict it back, minimize MSE.

In [ ]:
def p_losses(x0, t):
    noise    = torch.randn_like(x0)
    x_t      = q_sample(x0, t, noise)
    pred     = model(x_t, t)
    return F.mse_loss(pred, noise)


loss_history = []

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0

    pbar = tqdm(loader, desc=f'Epoch {epoch}/{epochs}', leave=False)
    for x0, _ in pbar:
        x0 = x0.to(device)
        t  = torch.randint(0, T, (x0.shape[0],), device=device)

        loss = p_losses(x0, t)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg = epoch_loss / len(loader)
    loss_history.append(avg)
    print(f'Epoch {epoch:2d} | loss {avg:.4f}')

plt.figure(figsize=(7, 3))
plt.plot(loss_history, color='#8f3e22', linewidth=2, marker='o', markersize=4)
plt.xlabel('epoch')
plt.ylabel('MSE loss')
plt.title('Training loss — denoising objective')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5: DDPM Sampling — 1000 Steps

The reverse process (Algorithm 2 from Ho et al.):

$$\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\epsilon_\theta(x_t, t)\right)$$

$$x_{t-1} = \mu_\theta + \sqrt{\beta_t}\,z, \quad z \sim \mathcal{N}(0, I)\text{ (for } t > 1\text{)}$$

In [ ]:
@torch.no_grad()
def p_sample_ddpm(x_T, n_samples=16):
    """DDPM reverse process — 1000 steps."""
    x = x_T
    model.eval()

    for t_val in tqdm(reversed(range(T)), total=T, desc='DDPM sampling', leave=False):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)

        eps_pred = model(x, t_batch)

        # Compute mean μ_θ
        alpha_t   = extract(alphas,    t_batch, x.shape)
        ab_t      = extract(alpha_bar, t_batch, x.shape)
        beta_t    = extract(betas,     t_batch, x.shape)

        mu = (1.0 / alpha_t.sqrt()) * (x - beta_t / (1 - ab_t).sqrt() * eps_pred)

        if t_val > 0:
            z = torch.randn_like(x)
            x = mu + beta_t.sqrt() * z
        else:
            x = mu

    return x


import time
n_samples = 16
x_T = torch.randn(n_samples, 1, img_size, img_size, device=device)

t0      = time.time()
samples = p_sample_ddpm(x_T, n_samples)
elapsed = time.time() - t0
print(f'DDPM sampling (1000 steps, {n_samples} samples): {elapsed:.1f}s')

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flatten()):
    img = samples[i, 0].cpu().clamp(-1, 1).numpy()
    ax.imshow((img + 1) / 2, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle(f'DDPM-generated MNIST digits (1000 steps, {elapsed:.1f}s)', y=1.02)
plt.tight_layout()
plt.show()

## Part 6: Visualise the Reverse Trajectory

Watch a single digit emerge from noise across the 1000 denoising steps.

In [ ]:
@torch.no_grad()
def p_sample_ddpm_trajectory(x_T, capture_steps=None):
    """Run DDPM and capture intermediate images at specified timesteps."""
    if capture_steps is None:
        capture_steps = [999, 900, 750, 500, 250, 100, 50, 10, 0]
    x       = x_T[:1]   # single sample
    captured = {}
    model.eval()

    for t_val in reversed(range(T)):
        t_batch  = torch.full((1,), t_val, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        alpha_t  = extract(alphas,    t_batch, x.shape)
        ab_t     = extract(alpha_bar, t_batch, x.shape)
        beta_t   = extract(betas,     t_batch, x.shape)
        mu = (1.0 / alpha_t.sqrt()) * (x - beta_t / (1 - ab_t).sqrt() * eps_pred)
        x  = mu + (beta_t.sqrt() * torch.randn_like(x) if t_val > 0 else 0)
        if t_val in capture_steps:
            captured[t_val] = x[0, 0].cpu().clamp(-1, 1).numpy()

    return captured


capture_steps = [999, 900, 750, 500, 250, 100, 50, 10, 0]
traj = p_sample_ddpm_trajectory(x_T, capture_steps)

fig, axes = plt.subplots(1, len(capture_steps), figsize=(14, 2))
for ax, step in zip(axes, capture_steps):
    img = traj[step]
    ax.imshow((img + 1) / 2, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f't={step}', fontsize=8)
    ax.axis('off')
plt.suptitle('Reverse denoising trajectory — noise → digit', y=1.05)
plt.tight_layout()
plt.show()

## Summary

We implemented DDPM from scratch:

| Component | Code | Math |
|---|---|---|
| Noise schedule | `betas = linspace(1e-4, 0.02, T)` | $\beta_1 < \cdots < \beta_T$ |
| Forward process | `q_sample(x0, t, noise)` | $q(x_t|x_0) = \mathcal{N}(\sqrt{\bar\alpha_t}x_0, (1-\bar\alpha_t)I)$ |
| Denoising loss | `F.mse_loss(model(x_t, t), noise)` | $\mathcal{L} = \mathbb{E}[\|\epsilon - \epsilon_\theta(x_t, t)\|^2]$ |
| DDPM sampling | `p_sample_ddpm(x_T)` | 1000-step reverse SDE |

**Next:** `demo_02_ddim_sampling.ipynb` — same model, 50 steps, 20× faster.